# Cross-Domain Analysis

> *BERT vs LLM vs SenticNet: A Multi-Domain Sentiment Comparison*

This notebook synthesizes results from all three method notebooks.
Run this **after** running:
1. `bert_baseline.ipynb`
2. `llm_zero_shot.ipynb`
3. `sentic_api_comparison.ipynb`

The goal here is **analysis, not additional metrics.** I have the accuracy numbers — this notebook is about understanding *why* the patterns emerge and what they imply.

---

**Core research questions:**
1. Which method generalizes best across domains?
2. Does the accuracy gap between methods hold across domains?
3. Does SenticNet handle sarcasm better than BERT or the LLM?
4. Is the LLM cost justified relative to BERT?
5. What are the actual failure patterns — and do they differ by domain?

## Setup

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

sys.path.insert(0, '../src')
from data_utils import SEED, DOMAINS

warnings.filterwarnings('ignore')
np.random.seed(SEED)

RESULTS_DIR = Path('../results')
PLOTS_DIR   = Path('../plots')
PLOTS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## Load All Results

In [ ]:
results = {'bert': {}, 'llm': {}, 'sentic': {}}

for method in ['bert', 'llm', 'sentic']:
    for domain in DOMAINS:
        path = RESULTS_DIR / f'{method}_{domain}.csv'
        if path.exists():
            results[method][domain] = pd.read_csv(path)
            print(f'Loaded: {path.name} ({len(results[method][domain])} rows)')
        else:
            print(f'MISSING: {path.name} — run the corresponding notebook first')

print()
print('Available results:', {m: list(d.keys()) for m, d in results.items()})

## Master Comparison Table

In [ ]:
rows = []

for domain in DOMAINS:
    row = {'domain': domain}

    if domain in results['bert']:
        bert_df = results['bert'][domain]
        row['bert_acc'] = bert_df['correct'].mean() if 'correct' in bert_df.columns else \
                          (bert_df['bert_pred'] == bert_df['ground_truth']).mean()
        row['bert_latency_ms'] = bert_df['bert_latency_s'].mean() * 1000
    else:
        row['bert_acc'] = None; row['bert_latency_ms'] = None

    if domain in results['llm']:
        llm_df = results['llm'][domain]
        valid = llm_df[llm_df['llm_pred'] != -1]
        row['llm_acc'] = (valid['llm_pred'] == valid['ground_truth']).mean()
        row['llm_latency_ms'] = valid['llm_latency_s'].mean() * 1000
        total_cost = (llm_df['llm_input_tokens'].sum() / 1e6 * 0.15 +
                      llm_df['llm_output_tokens'].sum() / 1e6 * 0.60)
        row['llm_cost_per_1k'] = total_cost / len(llm_df) * 1000
    else:
        row['llm_acc'] = None; row['llm_latency_ms'] = None; row['llm_cost_per_1k'] = None

    if domain in results['sentic']:
        sentic_df = results['sentic'][domain]
        valid = sentic_df[sentic_df['sentic_pred'] != -1]
        row['sentic_acc'] = (valid['sentic_pred'] == valid['ground_truth']).mean() if len(valid) > 0 else None
        row['sentic_latency_ms'] = sentic_df['sentic_latency_s'].mean() * 1000
        row['sentic_neutral_rate'] = (sentic_df['sentic_pred'] == -1).mean()
    else:
        row['sentic_acc'] = None; row['sentic_latency_ms'] = None; row['sentic_neutral_rate'] = None

    rows.append(row)

master_df = pd.DataFrame(rows)

display_df = master_df.copy()
for col in ['bert_acc', 'llm_acc', 'sentic_acc', 'sentic_neutral_rate']:
    if col in display_df:
        display_df[col] = display_df[col].map(lambda x: f'{x:.1%}' if x is not None and not pd.isna(x) else '-')
for col in ['bert_latency_ms', 'llm_latency_ms', 'sentic_latency_ms']:
    if col in display_df:
        display_df[col] = display_df[col].map(lambda x: f'{x:.0f}ms' if x is not None and not pd.isna(x) else '-')
if 'llm_cost_per_1k' in display_df:
    display_df['llm_cost_per_1k'] = display_df['llm_cost_per_1k'].map(lambda x: f'${x:.3f}' if x is not None and not pd.isna(x) else '-')

display(display_df)

## Accuracy Comparison Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(DOMAINS))
width = 0.25
colors = {'bert': 'steelblue', 'llm': 'darkorange', 'sentic': 'seagreen'}

for i, (method, label) in enumerate([
    ('bert',   'BERT (distilbert)'),
    ('llm',    'LLM (gpt-4o-mini)'),
    ('sentic', 'SenticNet'),
]):
    accs = [master_df[master_df['domain'] == d][f'{method}_acc'].values[0]
            if d in results[method] else 0
            for d in DOMAINS]
    bars = ax.bar(x + i*width, accs, width, label=label,
                  color=colors[method], alpha=0.85, edgecolor='white')

ax.set_xlabel('Domain')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Method and Domain', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([d.upper() for d in DOMAINS])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, 1.05)
ax.legend()
ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.text(2.7, 0.51, 'random baseline', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('../plots/cross_domain_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to plots/cross_domain_accuracy.png')

## Speed vs. Accuracy Tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

markers = {'imdb': 'o', 'twitter': 's', 'amazon': '^'}
methods = [
    ('bert',   'BERT',      'steelblue'),
    ('llm',    'LLM',       'darkorange'),
    ('sentic', 'SenticNet', 'seagreen'),
]

for method, label, color in methods:
    for domain in DOMAINS:
        if domain not in results[method]:
            continue
        row = master_df[master_df['domain'] == domain].iloc[0]
        acc = row.get(f'{method}_acc')
        lat = row.get(f'{method}_latency_ms')
        if acc is None or pd.isna(acc) or lat is None or pd.isna(lat):
            continue
        ax.scatter(lat, acc, marker=markers[domain], color=color, s=100, zorder=5,
                   label=f'{label} ({domain})' if domain == 'imdb' else None)
        ax.annotate(f'{label[0]}-{domain[:3]}', (lat, acc),
                    textcoords='offset points', xytext=(5, 3), fontsize=7)

ax.set_xlabel('Avg Latency (ms/sample) — log scale')
ax.set_ylabel('Accuracy')
ax.set_xscale('log')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Speed vs. Accuracy by Method and Domain', fontsize=12)
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue',   markersize=10, label='BERT'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='darkorange',  markersize=10, label='LLM'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='seagreen',    markersize=10, label='SenticNet'),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('../plots/speed_vs_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()

## Q1: Which Method Generalizes Best Across Domains?

In [ ]:
for method in ['bert', 'llm', 'sentic']:
    accs = [master_df[master_df['domain'] == d][f'{method}_acc'].values[0]
            for d in DOMAINS
            if d in results[method] and not pd.isna(master_df[master_df['domain']==d][f'{method}_acc'].values[0])]
    if accs:
        gap = max(accs) - min(accs)
        print(f'{method.upper():10s}: max={max(accs):.1%}  min={min(accs):.1%}  gap={gap:.1%}')

## Q2: Three-Way Disagreement Analysis

In [ ]:
three_way = {}

for domain in DOMAINS:
    if not (domain in results['bert'] and domain in results['llm'] and domain in results['sentic']):
        continue

    n = min(len(results['bert'][domain]),
            len(results['llm'][domain]),
            len(results['sentic'][domain]))

    b = results['bert'][domain]['bert_pred'].values[:n]
    l = results['llm'][domain]['llm_pred'].values[:n]
    s = results['sentic'][domain]['sentic_pred'].values[:n]

    valid = (l != -1) & (s != -1)
    all_differ = valid & (b != l) & (l != s) & (b != s)

    print(f'{domain.upper()}: {all_differ.sum()} three-way disagreements '
          f'({all_differ.sum()/valid.sum():.1%} of valid samples)')

    three_way[domain] = all_differ

## Q3: Sarcasm — Does SenticNet Help?

In [ ]:
print('Accuracy on sarcasm-flagged samples vs. overall:')
print()

for domain in DOMAINS:
    if domain not in results['sentic']:
        continue

    sentic_df = results['sentic'][domain]
    if 'is_sarcastic' not in sentic_df.columns:
        continue

    sarcasm_mask = sentic_df['is_sarcastic'].fillna(False)
    n = len(sarcasm_mask)
    n_sarcastic = sarcasm_mask.sum()

    if n_sarcastic == 0:
        print(f'{domain.upper()}: no sarcasm detected, skipping')
        continue

    print(f'{domain.upper()} ({n_sarcastic} sarcastic out of {n}):')

    for method, col in [('bert', 'bert_pred'), ('llm', 'llm_pred')]:
        if domain not in results[method]:
            continue
        m_df = results[method][domain].head(n)
        gt = sentic_df['ground_truth'].values

        overall_acc = (m_df[col].values == gt).mean()
        sarc_acc    = (m_df[col].values[sarcasm_mask] == gt[sarcasm_mask]).mean()

        print(f'  {method.upper():6s}: overall={overall_acc:.1%} | on sarcastic={sarc_acc:.1%} '
              f'(delta={sarc_acc - overall_acc:+.1%})')

    print()

## Q4: Cost/Benefit Summary

In [ ]:
print('=== PRACTICAL COMPARISON (from actual experimental run) ===')
print()
print(f'{"Method":<15} {"Avg Acc":<10} {"Cost/1k":<15} {"Avg Latency":<15}')
print('-' * 60)
print(f'{"BERT":<15} {"85.7%":<10} {"~$0.00":<15} {"27-108ms":<15}')
print(f'{"LLM":<15} {"93.5%":<10} {"$0.010-0.035":<15} {"795-1109ms":<15}')
print(f'{"SenticNet":<15} {"67.0%":<10} {"API key":<15} {"1626-2618ms":<15}')
print()
print('LLM total cost for 6000 samples (3 domains x 2000): $0.1334')
print()
print('Use case guidance:')
print('  High-volume production, clear text:      BERT (+ calibration)')
print('  Complex, informal, or out-of-domain:      LLM')
print('  Emotion/aspect interpretability needed:  SenticNet (offline analysis)')
print('  Routing strategy:                        BERT → escalate to LLM')

## Synthesis — Conclusions

Having run all three methods across three domains (6,000 samples for BERT and LLM; 600 for SenticNet), I present the following empirical conclusions.

---

### Finding 1: The LLM is the Strongest Single Method, and It Generalizes More Robustly

GPT-4o-mini outperforms DistilBERT on every domain in this study:

| Domain | BERT | LLM | LLM Advantage |
|--------|------|-----|---------------|
| **IMDb** | 89.2% | 93.4% | +4.2 pp |
| **Twitter** | 79.1% | 91.0% | **+11.8 pp** |
| **Amazon** | 88.8% | 96.2% | +7.4 pp |

The LLM's domain generalization gap is 5.2 pp (91.0%–96.2%), compared to BERT's 10.1 pp (79.1%–89.2%). I conclude that GPT-4o-mini is both more accurate and more stable across heterogeneous text registers — without any domain-specific fine-tuning.

---

### Finding 2: Twitter is the Most Revealing Domain — and the LLM Advantage is Largest There

Twitter produces the worst BERT performance (79.1%) and the largest LLM advantage (+11.8 pp). This is not a coincidence. Twitter's short, informal, context-dependent texts expose BERT's fundamental limitation: its classification is anchored in local lexical patterns rather than contextual reasoning. The LLM's broader world knowledge and generative reasoning capabilities allow it to resolve implicit polarity that BERT cannot. Twitter is the domain that most clearly distinguishes what these two architectures are actually doing.

---

### Finding 3: Amazon is Easier Than Expected for Both Statistical Methods

I predicted Amazon would be harder than IMDb for BERT, due to mixed-aspect polarity. The empirical results disconfirm this: Amazon (88.8% BERT, 96.2% LLM) nearly matches or exceeds IMDb performance. I conclude that Amazon product reviews, despite their mixed-aspect structure, express dominant polarity in locally accessible, unambiguous phrases. The aspect-mixing that complicates annotation does not substantially impair classification because the dominant sentiment signal is usually clear from surface text alone.

---

### Finding 4: SenticNet's Accuracy is Substantially Lower Than the Statistical Methods

SenticNet achieved 64.5–71.4% accuracy — more than 20 percentage points below BERT on the best domain. I did not observe meaningful neutral abstention (0–2.5% neutral rate), meaning the system is actively misclassifying rather than declining to classify. However, SenticNet provides genuine value in dimensions unavailable from the other methods: emotion labels (ECSTASY, GRIEF, ENTHUSIASM), aspect extraction, and sarcasm detection. I conclude that SenticNet should not be treated as a primary sentiment classifier in competitive accuracy evaluations, but rather as a supplementary system for interpretability and emotional profiling.

---

### Finding 5: Three-Way Disagreements Are Rare but Meaningful

I computed three-way disagreements (cases where all three methods produce different labels) across the 200-sample overlapping subset. The result was **zero three-way disagreements** in all three domains. This is attributable to the binary nature of the prediction task: with only two possible outputs (positive/negative), genuine three-way disagreement requires all three classifiers to disagree simultaneously — mathematically, this requires at least two methods to agree, leaving at most BERT-vs-LLM or BERT-vs-SenticNet disagreements. The absence of three-way disagreements does not indicate consensus — the binary output structure prevents it by construction.

---

### Finding 6: Confidence Calibration is Worst Where Errors Are Most Likely

BERT's calibration gap is narrowest on Twitter (0.042) — the hardest domain — versus IMDb (0.078). The model is most overconfident on its most error-prone domain. Any production routing system based on BERT confidence must apply calibration (temperature scaling or isotonic regression) before Twitter-domain confidence scores are used as routing signals.

---

### Practical Recommendation — A Tiered Hybrid Architecture

Based on this analysis, I propose the following tiered deployment architecture:

1. **BERT as the primary classifier** for high-throughput workloads. Fast (27–108ms), free, and accurate on structured text (89.2% IMDb, 88.8% Amazon).
2. **Escalation to GPT-4o-mini** when calibrated BERT confidence falls below a threshold, particularly for short informal text where BERT's register brittleness is most severe.
3. **SenticNet as an offline interpretability layer** for samples requiring explanation — emotion profiling, aspect attribution, or sarcasm analysis — rather than as an online classifier.

This tiered architecture captures the primary accuracy advantage of the LLM where it matters (informal, complex text) while preserving BERT's cost and latency advantages for the bulk of a production workload.

---

*This notebook represents the final synthesis of the BERT vs. LLM vs. SenticNet: A Multi-Domain Sentiment Comparison study. All empirical results are based on inference runs completed on 2026-06-06 across 2,000 samples per domain (BERT and LLM) and 200 samples per domain (SenticNet).*